#LSTM TEXT CLASSIFICATION

In [1]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# Load dataset
vocab_size = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
# Padding
max_len = 200
X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

In [4]:
# Model
model = Sequential()
model.add(Embedding(vocab_size, 64, input_length=max_len))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Train
model.fit(X_train, y_train, epochs=3, batch_size=64, validation_data=(X_test, y_test))

Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 84s 206ms/step - accuracy: 0.7674 - loss: 0.4698 - val_accuracy: 0.8631 - val_loss: 0.3294
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 79s 198ms/step - accuracy: 0.8957 - loss: 0.2607 - val_accuracy: 0.8131 - val_loss: 0.5275
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 79s 203ms/step - accuracy: 0.9202 - loss: 0.2078 - val_accuracy: 0.8592 - val_loss: 0.3743


In [6]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model

In [7]:

# Load IMDB
vocab_size = 10000
(X_train, y_train), _ = imdb.load_data(num_words=vocab_size)


In [8]:
# Padding
max_len = 100
X_train = pad_sequences(X_train, maxlen=max_len)

In [9]:

# Convert labels to sequence
y_seq = np.repeat(y_train.reshape(-1,1), max_len, axis=1)
y_seq = np.expand_dims(y_seq, -1)


In [10]:
from tensorflow.keras.layers import Embedding

#encoding
encoder_inputs = Input(shape=(max_len,))
encoder_embed = Embedding(10000, 64)(encoder_inputs)

encoder = LSTM(64, return_state=True)
_, state_h, state_c = encoder(encoder_embed)

In [11]:
# Decoder
decoder_inputs = Input(shape=(max_len,1))
decoder_lstm = LSTM(64, return_sequences=True)
decoder_outputs = decoder_lstm(decoder_inputs, initial_state=[state_h, state_c])

decoder_dense = Dense(1, activation='sigmoid')
decoder_outputs = decoder_dense(decoder_outputs)

In [12]:
# Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='binary_crossentropy')

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 100, 64)   │    640,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 64),      │     33,024 │ embedding_1[0][0] │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 100, 64)   │     16,896 │ input_layer_2[0]… │
│                     │                   │            │ lstm_1[0][1],     │
│                     │                   │            │ lstm_1[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 100, 1)    │         65 │ lstm_2[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 689,985 (2.63 MB)

 Trainable params: 689,985 (2.63 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:

# Train
model.fit([X_train, y_seq], y_seq, epochs=2, batch_size=64)

sample_input = X_train[:1]
sample_target = y_seq[:1]

pred = model.predict([sample_input, sample_target])

print("Actual label sequence:", sample_target[0][:10])
print("Predicted sequence:", pred[0][:10])

Epoch 1/2
391/391 ━━━━━━━━━━━━━━━━━━━━ 56s 135ms/step - loss: 0.0665
Epoch 2/2
391/391 ━━━━━━━━━━━━━━━━━━━━ 53s 136ms/step - loss: 0.0074
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step
Actual label sequence: [[1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]]
Predicted sequence: [[0.87457144]
 [0.96971875]
 [0.995169  ]
 [0.9989911 ]
 [0.9996174 ]
 [0.99977523]
 [0.99982995]
 [0.999853  ]
 [0.9998641 ]
 [0.99987   ]]
